# Evolving a Minimal Neural Agent for Capture-the-Flag

This notebook is the final-project workspace for the CTRNN capture-the-flag model. The goal is to connect the code to the cognitive modeling question: how much adaptive behavior can emerge from a very small recurrent controller interacting with an environment?

## Project Question

Can a three-neuron continuous-time recurrent neural network evolve useful capture-the-flag behavior from limited ray-based perception and differential-drive action?

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
ROOT

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import toy_ctf
import mini_ctf

print('Stage 1 genome parameters:', toy_ctf.N_PARAMS)
print('Mini-CTF genome parameters:', mini_ctf.N_PARAMS)

## Model Components

- **Environment:** continuous 2D arena with flags, bases, and agents.
- **Sensors:** four directional rays with semantic channels.
- **Controller:** three-neuron CTRNN.
- **Actions:** left and right wheel speeds.
- **Learning:** genetic algorithm over controller parameters.

In [ ]:
# Smoke-test one random mini-CTF match.
rng = np.random.default_rng(0)
genome_a = rng.normal(size=mini_ctf.N_PARAMS)
genome_b = rng.normal(size=mini_ctf.N_PARAMS)

fit_a, fit_b, winner, trail = mini_ctf.run_match(genome_a, genome_b, seed=1, record=True)
fit_a, fit_b, winner, len(trail)

## Visualizing A Recorded Match

The trail records agent positions, heading, carrying state, and flag positions over time. This plot shows the movement paths from one match.

In [ ]:
arr = np.array(trail)

plt.figure(figsize=(6, 6))
plt.xlim(0, mini_ctf.ARENA)
plt.ylim(0, mini_ctf.ARENA)
plt.gca().set_aspect('equal')
plt.plot(arr[:, 0], arr[:, 1], label='Agent A')
plt.plot(arr[:, 4], arr[:, 5], label='Agent B')
plt.scatter([mini_ctf.base_pos(0)[0], mini_ctf.base_pos(1)[0]], [mini_ctf.base_pos(0)[1], mini_ctf.base_pos(1)[1]], marker='s', label='Bases')
plt.title(f'Random match paths, winner={winner}')
plt.legend()
plt.show()

## Running Evolution

For the final paper, use longer runs from the command line so the notebook stays readable. Example:

```powershell
& 'C:/Users/Nkuro/AppData/Local/Programs/Python/Python313/python.exe' mini_ctf.py --pop 40 --gens 100 --seed 0 --out best_mini_ctf.npy --no-animate
```

The script saves the final best genome, the all-time best genome, the hall of fame, and a fitness plot.

## Result Notes

Use this section to record what happened in each training run:

- Did best fitness improve?
- Did win rate become nonzero or reach 1.0?
- Did replay behavior show flag approach, pickup, return, or tagging?
- Did the final best differ from the all-time best?